# 01 Fit vs Complexity

Compare reconstruction metrics across methods and K/latent dimensions.

In [4]:
from pathlib import Path
import os
import sys
import json

def _find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for path in candidates:
        if (path / 'pyproject.toml').exists() and (path / 'src' / 'cytof_archetypes').exists():
            return path
    fallback = Path('/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv')
    if (fallback / 'src' / 'cytof_archetypes').exists():
        return fallback
    raise RuntimeError('Could not locate repository root containing src/cytof_archetypes')

REPO_ROOT = _find_repo_root()
SRC_DIR = REPO_ROOT / 'src'
def _resolve_out_dir() -> Path:
    env = os.environ.get('CYTOF_SUITE_OUTPUT_DIR')
    if env:
        return Path(env)
    cfg_path = REPO_ROOT / 'configs' / 'experiment_suite.yaml'
    if cfg_path.exists():
        try:
            import yaml
            cfg = yaml.safe_load(cfg_path.read_text(encoding='utf-8')) or {}
            out_raw = cfg.get('output_dir')
            if out_raw:
                out_path = Path(out_raw)
                return out_path if out_path.is_absolute() else REPO_ROOT / out_path
        except Exception:
            pass
    return REPO_ROOT / 'outputs' / 'experiment_suite'

OUT_DIR = _resolve_out_dir()

def _artifact_exists(path: Path) -> bool:
    if path.exists():
        return True
    print(f'Missing artifact: {path}')
    print('Run the suite first: python scripts/run_experiment_suite.py --config configs/experiment_suite.yaml')
    return False
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print('Repo root:', REPO_ROOT)
print('Using src dir:', SRC_DIR)
print('Using suite output dir:', OUT_DIR)


Repo root: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv
Using src dir: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/src
Using suite output dir: /Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/outputs/experiment_suite


In [5]:
import pandas as pd
from pathlib import Path
table_path = OUT_DIR / 'tables' / 'fit_vs_complexity_summary.csv'
if _artifact_exists(table_path):
    table = pd.read_csv(table_path)
    display(table.head())
else:
    table = None

,method,seed,k_or_latent_dim,val_mse,val_nll,test_mse,test_nll,ari,nmi,knn_purity,silhouette,param_count
0,nmf,13,8,0.409472,35.957581,0.389733,35.641758,0.027404,0.255308,0.522825,-0.201497,256
1,nmf,17,8,0.406981,35.917728,0.387428,35.604889,0.037261,0.298571,0.525669,-0.179515,256
2,nmf,23,8,0.406520,35.910358,0.386856,35.595718,0.034862,0.282551,0.524933,-0.188620,256
3,nmf,13,10,0.344912,34.924622,0.325195,34.609154,0.024041,0.237639,0.519378,-0.159899,320
4,nmf,17,10,0.344427,34.916859,0.325540,34.614677,0.023653,0.236987,0.515529,-0.168327,320


In [6]:
if table is not None:
    summary = table.groupby(['method', 'k_or_latent_dim'], as_index=False)[['val_mse','test_mse','test_nll']].mean()
    display(summary.sort_values(['method','k_or_latent_dim']).head(20))

,method,k_or_latent_dim,val_mse,test_mse,test_nll
0,ae,8,0.212606,0.207726,32.729656
1,ae,10,0.178038,0.173856,32.187728
2,ae,12,0.153507,0.150373,31.812005
3,ae,14,0.132005,0.127578,31.447276
4,ae,16,0.120900,0.118251,31.298042
5,classical_archetypes,8,0.578091,0.562664,38.408649
6,classical_archetypes,10,0.544397,0.528966,37.869483
7,classical_archetypes,12,0.525319,0.509624,37.560019
8,classical_archetypes,14,0.507946,0.493661,37.304607
9,classical_archetypes,16,0.497481,0.484542,37.158712


In [7]:
for fig in ['reconstruction_vs_k.png', 'nll_vs_k.png', 'parameter_count_vs_validation_error.png']:
    p = OUT_DIR / 'plots' / fig
    print(p, 'exists=', p.exists())

/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/outputs/experiment_suite/plots/reconstruction_vs_k.png exists= True
/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/outputs/experiment_suite/plots/nll_vs_k.png exists= True
/Users/ronguy/Dropbox/Work/CyTOF/Experiments/ProbAE_Deconv/outputs/experiment_suite/plots/parameter_count_vs_validation_error.png exists= True
